# Legacy vs. conformal wakefield simulation

This notebook compares the existing **legacy** STL discretization with the new
**conformal** geometry pipeline for the standard WAKIS cavity geometry.

Both simulations use exactly the same:

- cavity and shell STL files,
- Cartesian grid,
- beam parameters,
- wake length,
- PEC outer boundary conditions.

The material setup is:

- `cavity` → vacuum,
- `shell` → PEC,
- background → PEC.

The purpose is to compare the resulting wake potential and longitudinal impedance.
No analytical resonance frequencies are used because the cavity has smooth
pipe-to-cavity transitions and is not an ideal pillbox.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

from wakis import GridFIT3D, SolverFIT3D, WakeSolver

## Geometry and common simulation parameters

The notebook is assumed to live in the `notebooks` directory, so the existing
test STL files are addressed directly with relative paths.


In [ ]:
cavity_path = Path("../tests/stl/007_vacuum_cavity.stl")
shell_path = Path("../tests/stl/007_lossymetal_shell.stl")

stl_solids = {
    "cavity": str(cavity_path),
    "shell": str(shell_path),
}

stl_materials = {
    "cavity": "vacuum",
    "shell": "pec",
}

stl_material_types = {
    "cavity": "normal",
    "shell": "pec",
}

solids = pv.read(cavity_path) + pv.read(shell_path)
xmin, xmax, ymin, ymax, zmin, zmax = solids.bounds

# Same Cartesian mesh for both simulations.
Nx = 60
Ny = 60
Nz = 140

# Beam parameters.
sigmaz = 10e-2
q = 1e-9
beta = 1.0

xsource = 0.0
ysource = 0.0
xtest = 0.0
ytest = 0.0

# Wake calculation.
wakelength = 10.0
skip_cells = 20

# Closed PEC computational domain.
bc_low = ["pec", "pec", "pec"]
bc_high = ["pec", "pec", "pec"]

dtype = np.float32

results_root = Path("_conformal_pillbox_wake")
results_root.mkdir(exist_ok=True)

## Legacy simulation

The legacy and conformal cases both use `stl_method="voxelize_rectilinear"`.
This keeps the underlying STL classification method identical and isolates the
effect of the new conformal discretization.


In [ ]:
legacy_folder = results_root / "legacy"
legacy_folder.mkdir(parents=True, exist_ok=True)

legacy_ez_file = legacy_folder / "Ez.h5"
if legacy_ez_file.exists():
    legacy_ez_file.unlink()

grid_legacy = GridFIT3D(
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax,
    Nx,
    Ny,
    Nz,
    stl_solids=stl_solids,
    stl_materials=stl_materials,
    stl_material_types=stl_material_types,
    stl_method="voxelize_rectilinear",
    geometry_mode="legacy",
    subpixel_smoothing=False,
    stl_scale=1.0,
    stl_rotate=[0.0, 0.0, 0.0],
    stl_translate=[0.0, 0.0, 0.0],
    verbose=1,
)

wake_legacy = WakeSolver(
    wakelength=wakelength,
    q=q,
    sigmaz=sigmaz,
    beta=beta,
    xsource=xsource,
    ysource=ysource,
    xtest=xtest,
    ytest=ytest,
    save=False,
    Ez_file=str(legacy_ez_file),
    skip_cells=skip_cells,
    verbose=1,
)

solver_legacy = SolverFIT3D(
    grid_legacy,
    wake_legacy,
    bc_low=bc_low,
    bc_high=bc_high,
    use_stl=True,
    bg="pec",
    dtype=dtype,
    use_gpu=False,
    verbose=1,
)

solver_legacy.wakesolve(
    wakelength=wakelength,
    save_J=False,
)

## Conformal simulation

Only `geometry_mode` changes. The mesh, STL classification method, material
definitions, source and wake parameters remain identical.


In [ ]:
conformal_folder = results_root / "conformal"
conformal_folder.mkdir(parents=True, exist_ok=True)

conformal_ez_file = conformal_folder / "Ez.h5"
if conformal_ez_file.exists():
    conformal_ez_file.unlink()

grid_conformal = GridFIT3D(
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax,
    Nx,
    Ny,
    Nz,
    stl_solids=stl_solids,
    stl_materials=stl_materials,
    stl_material_types=stl_material_types,
    stl_method="voxelize_rectilinear",
    geometry_mode="conformal",
    subpixel_smoothing=False,
    stl_scale=1.0,
    stl_rotate=[0.0, 0.0, 0.0],
    stl_translate=[0.0, 0.0, 0.0],
    verbose=1,
)

wake_conformal = WakeSolver(
    wakelength=wakelength,
    q=q,
    sigmaz=sigmaz,
    beta=beta,
    xsource=xsource,
    ysource=ysource,
    xtest=xtest,
    ytest=ytest,
    save=False,
    Ez_file=str(conformal_ez_file),
    skip_cells=skip_cells,
    verbose=1,
)

solver_conformal = SolverFIT3D(
    grid_conformal,
    wake_conformal,
    bc_low=bc_low,
    bc_high=bc_high,
    use_stl=True,
    bg="pec",
    dtype=dtype,
    use_gpu=False,
    verbose=1,
)

solver_conformal.wakesolve(
    wakelength=wakelength,
    save_J=False,
)

## Basic numerical checks

Both simulations should produce wake and impedance arrays on compatible axes.
The time steps are also printed so that any difference in the time integration
is immediately visible.


In [ ]:
print(f"Legacy dt:    {solver_legacy.dt:.6e} s")
print(f"Conformal dt: {solver_conformal.dt:.6e} s")

print()
print(f"Legacy wake samples:    {len(wake_legacy.WP)}")
print(f"Conformal wake samples: {len(wake_conformal.WP)}")
print(f"Legacy impedance samples:    {len(wake_legacy.Z)}")
print(f"Conformal impedance samples: {len(wake_conformal.Z)}")

## Wake-potential comparison


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    wake_legacy.s,
    wake_legacy.WP,
    linewidth=1.4,
    label="Legacy",
)

ax.plot(
    wake_conformal.s,
    wake_conformal.WP,
    linewidth=1.4,
    linestyle="--",
    label="Conformal",
)

ax.set_xlabel("s [m]")
ax.set_ylabel("Wake potential")
ax.set_title("Pillbox cavity: longitudinal wake potential")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

## Longitudinal impedance comparison

The magnitude is the most convenient first comparison because shifts in the
cavity resonances become immediately visible.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    wake_legacy.f / 1e9,
    np.abs(wake_legacy.Z),
    linewidth=1.4,
    label="Legacy",
)

ax.plot(
    wake_conformal.f / 1e9,
    np.abs(wake_conformal.Z),
    linewidth=1.4,
    linestyle="--",
    label="Conformal",
)

ax.set_xlabel("Frequency [GHz]")
ax.set_ylabel(r"$|Z_\parallel|$ [$\Omega$]")
ax.set_title("Pillbox cavity: longitudinal impedance")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

## Real and imaginary parts

This plot helps distinguish an actual resonance-frequency shift from a change
only in peak magnitude.


In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(10, 8),
    sharex=True,
)

axes[0].plot(
    wake_legacy.f / 1e9,
    np.real(wake_legacy.Z),
    linewidth=1.3,
    label="Legacy",
)

axes[0].plot(
    wake_conformal.f / 1e9,
    np.real(wake_conformal.Z),
    linewidth=1.3,
    linestyle="--",
    label="Conformal",
)

axes[0].set_ylabel(r"Re$(Z_\parallel)$ [$\Omega$]")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(
    wake_legacy.f / 1e9,
    np.imag(wake_legacy.Z),
    linewidth=1.3,
    label="Legacy",
)

axes[1].plot(
    wake_conformal.f / 1e9,
    np.imag(wake_conformal.Z),
    linewidth=1.3,
    linestyle="--",
    label="Conformal",
)

axes[1].set_xlabel("Frequency [GHz]")
axes[1].set_ylabel(r"Im$(Z_\parallel)$ [$\Omega$]")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.suptitle("Pillbox cavity: longitudinal impedance")
fig.tight_layout()
plt.show()

## Relative differences

These diagnostics are normalized by the maximum magnitude of the legacy
solution rather than pointwise, so zeros in the wake or impedance do not
artificially produce huge relative errors.


In [ ]:
if np.array_equal(wake_legacy.s, wake_conformal.s):
    wake_difference = wake_conformal.WP - wake_legacy.WP
    wake_relative_difference = wake_difference / max(
        np.max(np.abs(wake_legacy.WP)), 1e-30
    )

    fig, ax = plt.subplots(figsize=(10, 4))

    ax.plot(
        wake_legacy.s,
        wake_relative_difference,
    )

    ax.set_xlabel("s [m]")
    ax.set_ylabel(
        r"$(W_\mathrm{conf}-W_\mathrm{legacy})/"
        r"\max|W_\mathrm{legacy}|$"
    )
    ax.set_title("Normalized wake-potential difference")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print("Wake axes differ; direct pointwise wake subtraction was skipped.")

In [ ]:
if np.array_equal(wake_legacy.f, wake_conformal.f):
    impedance_difference = np.abs(wake_conformal.Z) - np.abs(wake_legacy.Z)

    impedance_relative_difference = impedance_difference / max(
        np.max(np.abs(wake_legacy.Z)), 1e-30
    )

    fig, ax = plt.subplots(figsize=(10, 4))

    ax.plot(
        wake_legacy.f / 1e9,
        impedance_relative_difference,
    )

    ax.set_xlabel("Frequency [GHz]")
    ax.set_ylabel(
        r"$(|Z|_\mathrm{conf}-|Z|_\mathrm{legacy})/"
        r"\max|Z_\mathrm{legacy}|$"
    )
    ax.set_title("Normalized impedance-magnitude difference")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print("Frequency axes differ; direct pointwise impedance subtraction was skipped.")